# Training exposure: are the corpus's translations the ones in Tanzil / OPUS?

**Why.** OPUS has distributed the Tanzil collection of Qur'an translations since 2018, and OPUS is a
common source of training data for multilingual encoders. If an encoder saw a language's Qur'an
translation in training, it may retrieve that translation better for reasons unrelated to how well it
represents the language. Four of the five languages at the retrieval floor are absent from Tanzil, so
this has to be measured, not assumed.

**What this notebook does (CPU, about 5 minutes).**
1. For each of the 23 corpus languages (24 files, Bosnian twice) that Tanzil v1 includes, it downloads OPUS's Tanzil file
   for that language and compares every Tanzil translation with the corpus's translation. The
   measure is text containment: the share of the corpus translation's 12-character shingles, taken
   over letters only, that also occur in the Tanzil translation.
2. It classifies each language as **absent** from Tanzil, **same translation** (containment of
   0.5 or more), or **different translation** (0.2 or less), with anything in between reported as
   related.
3. It tests whether that exposure explains per-language retrieval beyond Joshi resource tier:
   Freedman–Lane permutation, 20,000 permutations, n = 40, pooled and per encoder.

These thresholds were fixed before running. A re-edition of the same translation typically shares
most of its long shingles; independent translations share only stock phrases.

The setup cell asks for three files **one at a time**: `translations.zip`,
`retrieval_per_language.csv` and `joint_language_table.csv`. Tanzil downloads from OPUS.

In [ ]:
import os, re, io, csv, glob, json, html, zipfile, unicodedata, urllib.request
from collections import Counter
import numpy as np, pandas as pd
from scipy import stats

OUT = "results_overlap"; os.makedirs(OUT, exist_ok=True)
NEED = {"translations.zip": "Quran folder, or anon_repo/data/",
        "retrieval_per_language.csv": "Quran folder, or anon_repo/results/",
        "joint_language_table.csv": "Quran folder, or anon_repo/data/metadata/"}

def ask_for(name, where):
    from google.colab import files
    while not os.path.exists(name):
        print(f"\nUpload  {name}   (found in: {where})")
        up = files.upload()
        if not up:
            print("  nothing uploaded, asking again"); continue
        got = list(up)[0]; stem, ext = os.path.splitext(name)
        if got != name and got.startswith(stem) and got.endswith(ext):
            os.replace(got, name)
        elif got != name:
            print(f"  got {got}, expected {name}; ignoring it and asking again"); os.remove(got); continue
        print(f"  ok: {name} ({os.path.getsize(name):,} bytes)")

try:
    import google.colab  # noqa: F401
    for name, where in NEED.items():
        if not os.path.exists(name): ask_for(name, where)
except ImportError:
    pass
for f in NEED: assert os.path.exists(f), f"missing {f}"

# OPUS Tanzil v1 raw files, from OPUS's own statistics.yaml
TANZIL_URL = {
 "albanian": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/sq.zip",
 "azeri": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/az.zip",
 "bosnian-rwwad": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/bs.zip",
 "bosnian-mihanovich": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/bs.zip",
 "chinese": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/zh.zip",
 "dutch": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/nl.zip",
 "english": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/en.zip",
 "french": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/fr.zip",
 "german": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/de.zip",
 "hausa": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ha.zip",
 "indonesian": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/id.zip",
 "japanese": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ja.zip",
 "kurdish": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ku.zip",
 "malayalam": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ml.zip",
 "persian": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/fa.zip",
 "portuguese": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/pt.zip",
 "romanian": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ro.zip",
 "spanish": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/es.zip",
 "tajik": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/tg.zip",
 "tamil": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ta.zip",
 "turkish": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/tr.zip",
 "urdu": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ur.zip",
 "uyghur": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/ug.zip",
 "uzbek": "https://object.pouta.csc.fi/OPUS-Tanzil/v1/raw/uz.zip"
}
# the 17 corpus languages Tanzil v1 does not include (OPUS info.yaml, 42 languages)
ABSENT = ["asante", "assamese", "croatian", "gujarati", "kannada", "khmer", "kyrgyz", "lingala",
          "lithuanian", "macedonian", "moore", "oromo", "pashto", "serbian", "tagalog", "vietnamese", "yoruba"]

## 1. The corpus translations

In [ ]:
if not os.path.exists("corpus"):
    with zipfile.ZipFile("translations.zip") as z: z.extractall("corpus")
csv_files = sorted(glob.glob("corpus/**/*.csv", recursive=True))

def load_csv(fp):
    keys = {}
    with open(fp, encoding="utf-8", errors="replace") as f:
        rdr = csv.reader(f); started = False
        for row in rdr:
            if not started:
                low = [(x or "").strip().lower() for x in row]
                if "sura" in low and "aya" in low:
                    started = True; iS = low.index("sura"); iA = low.index("aya")
                    iT = low.index("translation") if "translation" in low else 3
                continue
            try: keys[(int(row[iS]), int(row[iA]))] = row[iT]
            except Exception: pass
    return keys

lang = lambda fp: os.path.basename(fp).split("_")[0]
cnt = Counter(lang(f) for f in csv_files)
disp = {f: (lang(f) if cnt[lang(f)] == 1 else lang(f) + "-" + os.path.basename(f).split("_")[1]) for f in csv_files}
TEXT = {disp[f]: load_csv(f) for f in csv_files}
IDENT = {disp[f]: os.path.basename(f).split("-csv")[0] for f in csv_files}
assert len(TEXT) == 41, len(TEXT)
assert set(TANZIL_URL) | set(ABSENT) == set(TEXT), set(TEXT) ^ (set(TANZIL_URL) | set(ABSENT))
print(len(TEXT), "translation files;", len(TANZIL_URL), "in a Tanzil language,", len(ABSENT), "not")

## 2. Containment against every Tanzil translation of the same language

In [ ]:
K = 12
def letters(t):
    t = unicodedata.normalize("NFKC", html.unescape(t)).casefold()
    return "".join(ch for ch in t if unicodedata.category(ch)[0] in "LM")

def shingles(texts):
    out = set()
    for t in texts:
        s = letters(t)
        out.update(hash(s[i:i + K]) for i in range(len(s) - K + 1))
    return out

def tanzil_docs(url):
    fp = os.path.join("tanzil", os.path.basename(url))
    os.makedirs("tanzil", exist_ok=True)
    if not os.path.exists(fp): urllib.request.urlretrieve(url, fp)
    docs = {}
    with zipfile.ZipFile(fp) as z:
        for n in z.namelist():
            if not n.endswith(".xml"): continue
            raw = z.read(n).decode("utf-8", errors="replace")
            sents = re.findall(r"<s\b[^>]*>(.*?)</s>", raw, flags=re.S) or [re.sub(r"<[^>]+>", " ", raw)]
            docs[os.path.basename(n)] = [re.sub(r"<[^>]+>", " ", s) for s in sents]
    return docs

rows = []
for l, url in TANZIL_URL.items():
    ours = shingles(TEXT[l].values())
    for doc, sents in tanzil_docs(url).items():
        theirs = shingles(sents)
        rows.append({"language": l, "corpus_translation": IDENT[l], "tanzil_document": doc,
                     "tanzil_units": len(sents),
                     "containment": len(ours & theirs) / max(1, len(ours)),
                     "reverse_containment": len(ours & theirs) / max(1, len(theirs))})
OV = pd.DataFrame(rows)
OV.to_csv(f"{OUT}/containment_all_documents.csv", index=False)
BEST = OV.sort_values("containment", ascending=False).groupby("language").head(1).set_index("language")
def verdict(c): return "same translation" if c >= 0.5 else ("different translation" if c <= 0.2 else "related (between thresholds)")
BEST["verdict"] = BEST.containment.map(verdict)
BEST = BEST.sort_values("containment", ascending=False)
BEST.round(3).to_csv(f"{OUT}/overlap_by_language.csv")
print(BEST[["corpus_translation", "tanzil_document", "tanzil_units", "containment", "reverse_containment", "verdict"]].round(3).to_string())
print("\n", BEST.verdict.value_counts().to_string())

## 3. Does exposure explain retrieval beyond resource tier?

n = 40 (one Bosnian translation, `bosnian-rwwad`, as in the paper). Three covariates, each tested
beyond Joshi tier: *in Tanzil* (the language appears there), *same translation* (the corpus
translation itself appears there), and, among the 23 languages Tanzil includes, *same versus different
translation*.

In [ ]:
ENC = {"sentence-transformers/LaBSE": "labse", "BAAI/bge-m3": "bgem3", "intfloat/multilingual-e5-base": "e5",
       "sentence-transformers/paraphrase-multilingual-mpnet-base-v2": "mpnet"}
R = pd.read_csv("retrieval_per_language.csv")
P = R.assign(e=R.model.map(ENC)).dropna(subset=["e"]).pivot(index="language", columns="e", values="mean_top1")
J = pd.read_csv("joint_language_table.csv").set_index("language")
D = J.join(P).drop(index="bosnian-mihanovich")
D["pooled"] = D[["labse", "bgem3", "e5", "mpnet"]].mean(axis=1)
D["in_tanzil"] = [0.0 if l in ABSENT else 1.0 for l in D.index]
D["same_translation"] = [1.0 if (l in BEST.index and BEST.loc[l, "verdict"] == "same translation") else 0.0 for l in D.index]
assert len(D) == 40

rng = np.random.default_rng(42)
def ols(y, X):
    X = np.column_stack([np.ones(len(y)), X]); b, *_ = np.linalg.lstsq(X, y, rcond=None); r = y - X @ b
    return 1 - (r @ r) / ((y - y.mean()) @ (y - y.mean())), b, X @ b, r
def freedman_lane(y, tier, x, n=20000):
    r2r, _, fit, res = ols(y, tier[:, None]); r2f, b, _, _ = ols(y, np.column_stack([tier, x]))
    null = np.array([ols(ys, np.column_stack([tier, x]))[0] - ols(ys, tier[:, None])[0]
                     for ys in (fit + res[rng.permutation(len(y))] for _ in range(n))])
    return {"R2_tier": r2r, "dR2": r2f - r2r, "coef": b[2], "p": float((null >= r2f - r2r).mean())}
def boot_coef(y, tier, x, n=10000):
    out = []; g = np.random.default_rng(7)
    for _ in range(n):
        i = g.integers(0, len(y), len(y))
        if x[i].min() == x[i].max(): continue
        out.append(ols(y[i], np.column_stack([tier[i], x[i]]))[1][2])
    return np.percentile(out, [2.5, 97.5])

rows = []
for cov, sub in [("in_tanzil", D), ("same_translation", D), ("same_vs_different (Tanzil languages)", D[D.in_tanzil == 1])]:
    x_col = "same_translation" if cov.startswith("same") else cov
    for col in ["pooled", "labse", "bgem3", "e5", "mpnet"]:
        y, t, x = sub[col].values, sub.joshi.values.astype(float), sub[x_col].values
        if x.min() == x.max(): continue
        r = freedman_lane(y, t, x); lo, hi = boot_coef(y, t, x) if col == "pooled" else (np.nan, np.nan)
        rows.append({"covariate": cov, "retrieval": col, "n": len(sub), "n_exposed": int(x.sum()),
                     **{k: round(v, 4) for k, v in r.items()}, "coef_lo": round(lo, 3), "coef_hi": round(hi, 3)})
REG = pd.DataFrame(rows); REG.to_csv(f"{OUT}/exposure_regression.csv", index=False)
print(REG.to_string(index=False))

In [ ]:
S = {"n_corpus_files_in_tanzil_languages": len(TANZIL_URL), "verdicts": BEST.verdict.value_counts().to_dict(),
     "same_translation": BEST[BEST.verdict == "same translation"].index.tolist(),
     "related": BEST[BEST.verdict.str.startswith("related")].index.tolist(),
     "regression": REG.to_dict(orient="records")}
json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=float)
print(json.dumps({k: v for k, v in S.items() if k != "regression"}, indent=1))
import shutil; shutil.make_archive("results_overlap", "zip", OUT)
try:
    from google.colab import files; files.download("results_overlap.zip")
except Exception:
    print("written results_overlap.zip")

## Reading the results

| Result | Meaning |
|---|---|
| Most Tanzil languages have a **different translation** in the corpus | Memorization of the exact texts is unlikely even where the language overlaps. |
| **same translation** covariate adds no significant variance beyond tier, and its coefficient is small against the floor gap (about 0.4 top-1) | Exposure to these particular texts does not explain the floor. |
| A significant, positive coefficient | Part of the floor tracks exposure. Report its size; the tokenizer axis does not depend on exposure, so the dissociation stands either way. |